In [2]:
from glob import glob
import os 
import re
import copy
import itertools
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter, NullFormatter
from PIL import Image
import json

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression, Lasso, LassoCV, Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler

In [ ]:
import sklearn
print ("numpy: ", np.__version__)
print ("pandas: ", pd.__version__)
print ("sklearn: ", sklearn.__version__)

In [1]:
# Root of the analysis tree (data/, regression/, result/).
# Unlike the earlier revision this is already relative; keep it that way, or set
# it to the absolute path of your local copy.
path = "./"


In [5]:
PROPERTY_NAMES = [
    "degradation",
    "elongation_break",
    "young_modulus",
    "crystallinity"
]

In [6]:
Reg_list=["Linear", "Ridge", "Lasso"]
Measurement_list=["all", "IR", "WAXS", "POM"]
# These strings are used as the *column suffix* of the CSVs written by the
# Fig. 4 pipeline ("Real,CryDeg", "Mul,CryDeg", ...) and as the figure titles.
# They must match the producer exactly, or the read_csv lookups raise KeyError.
Property_list=["Deg", "Strain", "Young", "CryDeg"]
model_list=["PG", "AE", "VAE", "CL", "CL2"]
color_list1=["lightblue", "lightcoral", "lightgreen", "moccasin", "plum"]
color_list2=["darkblue", "darkred", "darkgreen", "darkorange", "indigo"]
latent_list=[2, 4, 8, 16, 32, 64, 128, 256, 512]

In [7]:
plt.rcParams["font.family"] = "Arial"   
plt.rcParams["figure.dpi"] = 120

# Figure2

In [ ]:
df_IR=pd.read_csv(path+'/data/measurement/IR.csv', index_col=0)
df_WAXS=pd.read_csv(path+"/data/measurement/WAXS.csv", index_col=0)
POM_list=np.load(path+'/data/measurement/POM2.npy')
df_IR.shape, df_WAXS.shape, POM_list.shape

In [ ]:
for i in [0,66,45]:
    plt.figure(figsize=(4,2))
    plt.plot(df_IR.columns.astype("float"), df_IR.iloc[i].astype("float"), color="black")
    plt.gca().invert_xaxis()
    if i==45:
        plt.xlabel("Wavenumber / cm$^{-1}$", fontsize=16)
        plt.tick_params(axis='x', labelsize=14)
        plt.yticks([0,0.3,0.6,0.9]) 
    else:
        plt.tick_params(axis='x', labelbottom=False)
        plt.yticks([0,0.3,0.6]) 
    plt.tick_params(axis='y', labelsize=14)
    
    plt.ylabel("Absorbance", fontsize=16)
    plt.show()

In [ ]:
for i in [0,66,45]:
    plt.figure(figsize=(4,2))
    plt.plot(df_WAXS.columns.astype("float"), df_WAXS.iloc[i].astype("float"), color="black")
    
    if i==45:
        plt.xlabel("$q$ / nm$^{-1}$", fontsize=16)
        plt.tick_params(axis='x', labelsize=14)
        plt.xticks([5,10,15,20,25]) 
    else:
        plt.tick_params(axis='x', labelbottom=False)
    plt.yticks([]) 
    plt.ylabel("$I$($q$)", fontsize=16)
    plt.show()

In [ ]:
for i in [0,66,45]:
    plt.figure(figsize=(4,2.5))
    plt.imshow(POM_list[i], cmap='gray')
    
    plt.axis('off') 
    plt.show()


In [ ]:
df_param=pd.read_csv(path+'/data/params/param_sum.csv', index_col=0)
df_dc=df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9']].sum(axis=1)/df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9','WAXS_Int10','WAXS_Int11']].sum(axis=1)
df_param=pd.read_csv(path+'/data/params/param_sum.csv',index_col=0)
df_pred=pd.concat([df_param[['Property_Deg', 'Property_StrainBreak', 'Property_Young']], df_dc], axis=1)
df_pred['Property_Deg']=df_pred['Property_Deg']*100
df_pred.columns=["Deg","Strain","Young","CrysDeg"]
df_pred["CrysDeg"]=df_pred["CrysDeg"]*100
df_pred=df_pred.round({
    'Deg': 3,  
    'Strain': 4,  
    'Young': 0,  
    'CrysDeg': 2,  
})
df_pred

In [ ]:
label_list=["i)", "ii)", "iii)"]
plt.figure(figsize=(4,2))
plt.hist(df_pred["Deg"], bins=np.arange(0, 9.5, 0.5), rwidth=0.8, color="gray")
for j,i in enumerate([0,66,45]):
    plt.scatter(df_pred["Deg"].iloc[i], 1, color=color_list2[j], label=label_list[j])
plt.legend(fontsize=12)
plt.xlabel("Degradation rate (Deg) / %", fontsize=16)
plt.ylabel("Count", fontsize=16)
plt.yticks([0, 10, 20, 30])
plt.tick_params(labelsize=14)

In [ ]:
label_list=["i)", "ii)", "iii)"]
plt.figure(figsize=(4,2))
plt.hist(df_pred["Strain"], bins=np.arange(0, 0.18, 0.01), rwidth=0.8, color="gray")
for j,i in enumerate([0,66,45]):
    plt.scatter(df_pred["Strain"].iloc[i], 1, color=color_list2[j], label=label_list[j])
plt.legend(fontsize=12)
plt.xlabel("Strain at break (Strain)", fontsize=16)
plt.ylabel("Count", fontsize=16)
plt.yticks([0, 20, 40, 60])
plt.tick_params(labelsize=14)

In [ ]:
label_list=["i)", "ii)", "iii)"]
plt.figure(figsize=(4,2))
plt.hist(df_pred["Young"], bins=np.arange(1400, 2050, 50), rwidth=0.8, color="gray")
for j,i in enumerate([0,66,45]):
    plt.scatter(df_pred["Young"].iloc[i], 1, color=color_list2[j], label=label_list[j])
plt.legend(fontsize=12)
plt.xlabel("Young's modulus (Young) / MPa", fontsize=16)
plt.ylabel("Count", fontsize=16)
plt.yticks([0, 10, 20, 30])
plt.tick_params(labelsize=14)

In [ ]:
label_list=["i)", "ii)", "iii)"]
plt.figure(figsize=(4,2))
plt.hist(df_pred["CrysDeg"], bins=np.arange(0, 70, 5), rwidth=0.8, color="gray")
for j,i in enumerate([0,66,45]):
    plt.scatter(df_pred["CrysDeg"].iloc[i], 1, color=color_list2[j], label=label_list[j])
plt.legend(fontsize=12, bbox_to_anchor=(0.38,0.4))
plt.xlabel("Degree of crystallinity (CryDeg) / %", fontsize=15)
plt.ylabel("Count", fontsize=16)
plt.yticks([0, 10, 20, 30])
plt.tick_params(labelsize=14)

# Figure4

In [ ]:
count=0
for Measurement in ["Mul", "IR", "WAXS", "POM"]:
    for k, prop in enumerate(["Deg", "Strain", "Young", "CryDeg"]):
        if (Measurement!="Mul")&(prop!="Deg"):
            continue
        plt.figure(figsize=(3,3))
        r2_train_list, r2_test_list=[], []
        for latent in latent_list:
            r2_train, r2_test=[], []
            for seed in range(10):
                df_train=pd.read_csv(path+'/result/Analysis2/PG2/y_train_PG2_'+str(latent)+"_"+str(seed)+'.csv', index_col=0)
                df_test=pd.read_csv(path+'/result/Analysis2/PG2/y_test_PG2_'+str(latent)+"_"+str(seed)+'.csv', index_col=0)
                r2_train.append(r2_score(df_train['Real,'+prop], df_train[Measurement+','+prop]))
                r2_test.append(r2_score(df_test['Real,'+prop], df_test[Measurement+','+prop]))
            r2_train_list.append(r2_train)
            r2_test_list.append(r2_test)
        plt.errorbar(latent_list,
                     np.mean(np.array(r2_train_list), axis=1),
                     yerr=np.std(np.array(r2_train_list), axis=1),
                     fmt='o-', capsize=5, color="blue", ecolor="blue", label="train")
        plt.errorbar(latent_list,
                     np.mean(np.array(r2_test_list), axis=1),
                     yerr=np.std(np.array(r2_test_list), axis=1),
                     fmt='o-', capsize=5, color="red", ecolor="red", label="test")
        plt.xscale('log', base=2)
        plt.ylim(-0.1,1.05)
        if Measurement=="Mul":
            plt.title('All, '+Property_list[k], fontsize=16)
        else:
            plt.title(Measurement+', '+Property_list[k], fontsize=16)        
        if (Measurement=="Mul")&(prop=="Deg"):
            plt.legend(fontsize=18)
        if count in [0,4,5,6]:
            plt.xlabel("Latent Dim.", fontsize=18)
        if count in [0,1,4]:
            plt.ylabel("$R^2$", fontsize=18)
        plt.yticks([0, 0.5, 1])
        plt.tick_params(labelsize=16)
        ax = plt.gca()

        ax.xaxis.set_major_locator(FixedLocator([2, 32, 512]))
        ax.xaxis.set_major_formatter(FixedFormatter([r"$2^{1}$", r"$2^{5}$", r"$2^{9}$"]))
        
        ax.xaxis.set_minor_locator(FixedLocator([4, 8, 16, 64, 128, 256]))
        ax.yaxis.set_minor_locator(FixedLocator(np.arange(0.1,1.0,0.1)))
        ax.xaxis.set_minor_formatter(NullFormatter())
        plt.show()
        count+=1

# Figure5

In [ ]:
count=0
Reg="Lasso"
model_list=["PG1", "AE1", "VAE1", "CL1", "CL3"]
model_name=["PG", "AE", "VAE", "CL", "XCL"]

for j, Measurement in enumerate(Measurement_list):
    for i, Property in enumerate(Property_list):
        if (Measurement!="all")&(Property!="Deg"):
            continue
        plt.figure(figsize=(3,3))
        for color, model in enumerate(model_list):
            r2_train_list,r2_test_list=[],[]
            for No in range(10):
                df = pd.read_csv(path+"/regression/R2_"+model+"_"+str(No)+".csv", index_col=0)
                r2_train_list.append(df.loc[[Measurement+","+Reg+","+str(latent) for latent in latent_list], Property+",train"].values)
                r2_test_list.append(df.loc[[Measurement+","+Reg+","+str(latent) for latent in latent_list], Property+",test"].values)
            
            plt.errorbar(latent_list, np.mean(r2_test_list, axis=0), yerr=np.std(r2_test_list, axis=0),
                         fmt='o-', capsize=5, color=color_list2[color], ecolor=color_list2[color], label=model_name[color])
        plt.xscale('log', base=2)
        plt.ylim(-0.1,1.05)
        if Measurement=="all":
            plt.title('All, '+Property, fontsize=16)
        else:
            plt.title(Measurement+', '+Property, fontsize=16)
        if (Measurement=="all")&(Property=="Deg"):
            plt.legend(bbox_to_anchor=(0.65,0.59), fontsize=12)
        if count in [0,4,5,6]:
            plt.xlabel("Latent Dim.", fontsize=18)
        if count in [0,1,4]:
            plt.ylabel(r"$R^2_{\mathrm{test}}$", fontsize=18)
        plt.yticks([0, 0.5, 1])
        plt.tick_params(labelsize=16)
        ax = plt.gca()

        ax.xaxis.set_major_locator(FixedLocator([2, 32, 512]))
        ax.xaxis.set_major_formatter(FixedFormatter([r"$2^{1}$", r"$2^{5}$", r"$2^{9}$"]))
        
        ax.xaxis.set_minor_locator(FixedLocator([4, 8, 16, 64, 128, 256]))
        ax.yaxis.set_minor_locator(FixedLocator(np.arange(0.1,1.0,0.1)))
        ax.xaxis.set_minor_formatter(NullFormatter())
        plt.show()
        count+=1

# Figure6

In [ ]:
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

plt.figure(figsize=(4,3))
for i, latent in enumerate(latent_list):
    losses_list=[]
    for seed in range(10):
                # NOTE: these are *supercomputer job stdout logs*, not released data.
        # This revision reads the "wisteria" cluster path; the earlier revision
        # read "/code/スパコン/...". Neither is part of the artefacts, so the
        # panel cannot be reproduced as-is — see the README for a substitute.
        pattern = path + "/code/wisteria/Analysis7/A7-AE1-0-" + str(seed) + ".sh.*.out"
        filename = sorted(glob(pattern))
        if not filename:
            raise FileNotFoundError(
                "Figure 6 needs the training logs matching:\n  " + pattern
            )
    
        losses = []
        
        with open(filename[0], "r", encoding="utf-8") as f:
            for line in f:
                m = re.search(r"Epoch\s+(\d+)\s+\|\s+loss\s*=\s*([\d.]+)", line)
                if m:
                    epoch = int(m.group(1))
                    loss = float(m.group(2))
                    losses.append(loss)
        losses_list.append(np.array(losses).reshape(-1,500)[i])
    plt.plot(np.arange(len(np.array(losses_list).mean(axis=0))),
                 np.array(losses_list).mean(axis=0),
                 #yerr=np.array(losses_list).std(axis=0),
                 label=str(latent), color=colors[i])
    plt.yscale('log', base=10)
    plt.ylim(0.8, 3.5)
    plt.legend(fontsize=12, title="Latent Dim.", title_fontsize=14, labelspacing=0.3, ncol=2)
    plt.xlabel("Epoch", fontsize=18)
    plt.ylabel(r"$Loss_{\mathrm{ train}}$", fontsize=18)
    plt.tick_params(labelsize=16)
    
    ax = plt.gca()
    ax.yaxis.set_major_locator(FixedLocator([1, 2, 3]))
    ax.yaxis.set_major_formatter(FixedFormatter([1, 2, 3]))
    
    ax.yaxis.set_minor_locator(FixedLocator([0.9]))
    ax.yaxis.set_minor_formatter(NullFormatter())

# Figure7

In [ ]:
model_list=["PG1", "AE1", "VAE1", "CL1", "CL3"]
model_name=["PG", "AE", "VAE", "CL", "XCL"]
count=0

for Measurement in Measurement_list:
    prop_list=['degradation', 'strain', 'young_modulus', 'crystallinity']
    for k, prop in enumerate(prop_list):
        if (Measurement!="all")&(prop!="degradation"):
            continue
        plt.figure(figsize=(3,3))
        for l, model in enumerate(model_list):    
            r2_list3=[]
            for latent in latent_list:
                r2_list2=[]
                for seed in range(10):
                    with open(path+"/result/Analysis7/"+model+"/Analysis7-"+model+"_"+str(latent)+"_"+str(seed)+"_"+Measurement+"_true.json", "r", encoding="utf-8") as f:
                        result_true_all = json.load(f)    
                    with open(path+"/result/Analysis7/"+model+"/Analysis7-"+model+"_"+str(latent)+"_"+str(seed)+"_"+Measurement+"_pred.json", "r", encoding="utf-8") as f:
                        result_pred_all = json.load(f)
                        true_ave, pred_ave=[], []
                    for i in range(33):
                        true_list, pred_list=[], []
                        for j in range(100):
                            try:
                                value = result_true_all[i][j][prop]
                                true_list.append(result_true_all[i][j][prop])
                                pred_list.append(result_pred_all[i][j][prop])
                            except (KeyError, IndexError):
                                pass
        
                        if (len(true_list) > 3)&(len(pred_list) > 3):
                            true_ave.append(np.mean(true_list))
                            pred_ave.append(np.mean(pred_list))
                    r2_list2.append(r2_score(true_ave, pred_ave))
                r2_list3.append(r2_list2)

            plt.errorbar(latent_list,
                         np.mean(np.sort(np.array(r2_list3), axis=1)[:, :], axis=1),
                         yerr=np.std(np.sort(np.array(r2_list3), axis=1)[:, :], axis=1),
                             fmt='o-', capsize=5, color=color_list2[l], ecolor=color_list2[l], label=model_name[l])
        plt.xscale('log', base=2)
        plt.ylim(-0.1,1.05)
        if Measurement=="all":
            plt.title('All, '+Property_list[k], fontsize=16)
        else:
            plt.title(Measurement+', '+Property_list[k], fontsize=16)
        if (Measurement=="all")&(prop=="degradation"):
            plt.legend(bbox_to_anchor=(0.62,0.5), fontsize=10.5)
        if count in [0,4,5,6]:
            plt.xlabel("Latent Dim.", fontsize=18)
        if count in [0,1,4]:
            plt.ylabel(r"$R^2_{\mathrm{test}}$", fontsize=18)
        plt.yticks([0, 0.5, 1])
        plt.tick_params(labelsize=16)
        ax = plt.gca()

        ax.xaxis.set_major_locator(FixedLocator([2, 32, 512]))
        ax.xaxis.set_major_formatter(FixedFormatter([r"$2^{1}$", r"$2^{5}$", r"$2^{9}$"]))
        
        ax.xaxis.set_minor_locator(FixedLocator([4, 8, 16, 64, 128, 256]))
        ax.yaxis.set_minor_locator(FixedLocator(np.arange(0.1,1.0,0.1)))
        ax.xaxis.set_minor_formatter(NullFormatter())
        plt.show()
        count+=1

In [ ]:
# ---------------------------------------------------------------------------
# Saving figures (added)
# ---------------------------------------------------------------------------
# This notebook only calls plt.show(), so no figure file is ever written.
# Replace `plt.show()` with `save("Figure7_all_degradation")` to export.
import os

FIG_DIR = os.path.join(path, "figures")
os.makedirs(FIG_DIR, exist_ok=True)


def save(name, fig=None, dpi=300):
    """Write the current figure as <name>.pdf and <name>.png."""
    fig = fig or plt.gcf()
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=dpi,
                    bbox_inches="tight")
    print("saved", os.path.join(FIG_DIR, name))


# Reproducibility notes (added)

This is the *cleaned* copy of `Paper-figure_final.ipynb` (46 cells). It is the
short, publication-scoped revision: Figure 2, 4, 5, 6 and 7 only. The earlier,
longer revision (`Paper-figure_full.ipynb`, 103 cells) additionally contains
Figure 8 and the Analysis6 / 8–12 comparison panels and the 4x4 regressor grids.

Only the points below were changed; every plotting call and every data lookup is
untouched.

1. `path` is documented. This revision already used the relative `"./"`; the
   earlier one hard-coded a macOS absolute path.
2. A comment was added to `Property_list`: those strings are used as the
   **column suffix** of the CSVs written by the Fig. 4 pipeline and as the figure
   titles, so they must match the producer exactly.
3. Figure 6's `glob` over `/code/wisteria/Analysis7/...sh.*.out` now raises a
   clear `FileNotFoundError` when the pattern matches nothing, and documents that
   these files are **supercomputer job stdout logs** — not released artefacts.
4. A `save(name)` helper and a `figures/` directory were added; the original never
   called `savefig`, so no figure file could be produced from it.

Not changed (deliberate): every axis limit, tick locator, colour, title, the
`latent_list`, the error bars, and the aggregation of the 100 generated answers.

Points that still need attention (see the review document):

- Figure 6 is not reproducible: it reads supercomputer stdout logs.
- `plt.rcParams["font.family"] = "Arial"` falls back silently on systems without
  Arial (it is a proprietary font), changing the appearance of every figure.
- The Figure 2 crystallinity panel is labelled "(CryDeg)" while the DataFrame
  column is `CrysDeg` — the label and the data name disagree.
- The Figure 7 panels average **all 10 seeds**
  (`np.sort(x, axis=1)[:, :]`); the Analysis6-12 panels of the earlier revision
  drop the lowest and highest (`[:, 1:-1]`). The statistic differs between
  figures.
- R^2 is computed on the **mean of the 100 generated answers**, and the ground
  truth is averaged the same way, which shrinks its variance and inflates R^2.
- A specimen is skipped entirely when fewer than four of the 100 answers parse
  (`len(true_list) > 3`), so the sample size varies per panel and is never
  reported.
- `for i in range(33)` hard-codes the test-set size (11 groups x 3 replicates).
- The multimodal panels read `result/Analysis2/PG2/`, while the Fig. 4 notebook
  writes `PG3` in its second revision.
